# putEMG — Deep Learning Model Training & Evaluation

Trains and evaluates 5 CNN architectures on the putEMG gesture dataset, then retrains
the best-performing model on the full dataset to produce final deployment weights.

| # | Model | Description |
|---|-------|-------------|
| 1 | **EEGNet** | Depthwise separable CNN, original baseline |
| 2 | **ShallowConvNet** | Temporal + spatial conv, square/log nonlinearity |
| 3 | **DeepConvNet** | 4 stacked conv blocks, increasing filter depth |
| 4 | **CNN_LSTM** | Spatial CNN → temporal pooling → 2-layer LSTM |
| 5 | **EMG_TCN** | Spatial mixing + 4 dilated temporal conv blocks |

**Data splits:**
- 80% training set (further split 90/10 into train/val for early stopping)
- 20% held-out test set → final evaluation only

> **Prerequisites**: Run `preprocessing_driver.ipynb` first to generate the model-ready `.mat` file.

In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import models
from emg_datahandler import train, evaluate, evaluateFinal
from emg_datahandler import train_loader, dev_loader, test_loader, full_loader

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [3]:
# ── Training ──────────────────────────────────────────────────────────────────
MAX_EPOCHS = 50
PATIENCE   = 10     # early-stopping patience (epochs without improvement)
MIN_DELTA  = 0.002  # minimum improvement to reset patience

# ── Hyperparameters ───────────────────────────────────────────────────────────
DROPOUT = 0.25
LR      = 1e-3

# ── Weights output ────────────────────────────────────────────────────────────
WEIGHTS_DIR = 'weights'

---
## Training

Each model is trained on the training set with:
- **Early stopping** — stops when val accuracy doesn't improve by `MIN_DELTA` for `PATIENCE` epochs
- **ReduceLROnPlateau** — halves the LR when val accuracy plateaus for 5 epochs
- **Best-state checkpointing** — restores the best-val-acc weights before test evaluation

In [4]:
model_registry = {
    'EEGNet':         lambda: models.EEGNet(dropout_rate=DROPOUT),
    'ShallowConvNet': lambda: models.ShallowConvNet(dropout_rate=DROPOUT),
    'DeepConvNet':    lambda: models.DeepConvNet(dropout_rate=DROPOUT),
    'CNN_LSTM':       lambda: models.CNN_LSTM(dropout_rate=DROPOUT),
    'EMG_TCN':        lambda: models.EMG_TCN(dropout_rate=DROPOUT),
}

test_accs   = {}
best_states = {}   # stores best weights for every model

for name, build_fn in model_registry.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"{'='*60}")

    model     = build_fn().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                  patience=5, min_lr=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = float('-inf')
    best_state   = None
    bad_epochs   = 0

    for epoch in range(MAX_EPOCHS):
        tr_loss  = train(model, train_loader, criterion, optimizer, device)
        val_acc  = evaluate(model, dev_loader, device)
        curr_lr  = optimizer.param_groups[0]['lr']

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step(val_acc)

        if val_acc >= (best_val_acc - MIN_DELTA):
            bad_epochs = 0
        else:
            bad_epochs += 1

        print(f"  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  dev={val_acc*100:.2f}%  "
              f"best={best_val_acc*100:.2f}%  lr={curr_lr:.2e}")

        if bad_epochs >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}.")
            break

    model.load_state_dict(best_state)
    best_states[name] = best_state

    print(f"\n--- {name} — Test Set Evaluation ---")
    test_acc = evaluateFinal(model, test_loader, device)
    test_accs[name] = test_acc


  Training: EEGNet
  Epoch   1: loss=1.2007  dev=71.20%  best=71.20%  lr=1.00e-03
  Epoch   2: loss=0.8214  dev=77.08%  best=77.08%  lr=1.00e-03
  Epoch   3: loss=0.7067  dev=79.01%  best=79.01%  lr=1.00e-03
  Epoch   4: loss=0.6415  dev=81.64%  best=81.64%  lr=1.00e-03
  Epoch   5: loss=0.5981  dev=80.93%  best=81.64%  lr=1.00e-03
  Epoch   6: loss=0.5642  dev=81.74%  best=81.74%  lr=1.00e-03
  Epoch   7: loss=0.5385  dev=80.32%  best=81.74%  lr=1.00e-03
  Epoch   8: loss=0.5397  dev=82.56%  best=82.56%  lr=1.00e-03
  Epoch   9: loss=0.5215  dev=82.56%  best=82.56%  lr=1.00e-03
  Epoch  10: loss=0.4909  dev=85.60%  best=85.60%  lr=1.00e-03
  Epoch  11: loss=0.4848  dev=80.93%  best=85.60%  lr=1.00e-03
  Epoch  12: loss=0.4695  dev=82.86%  best=85.60%  lr=1.00e-03
  Epoch  13: loss=0.4585  dev=80.73%  best=85.60%  lr=1.00e-03
  Epoch  14: loss=0.4474  dev=84.89%  best=85.60%  lr=1.00e-03
  Epoch  15: loss=0.4417  dev=82.66%  best=85.60%  lr=1.00e-03
  Epoch  16: loss=0.4399  dev=84.99

KeyboardInterrupt: 

---
## Results Summary

In [ ]:
print(f"\n{'Model':<18} {'Test Acc':>10}")
print('-' * 30)
for name in model_registry:
    acc = test_accs.get(name, float('nan')) * 100
    print(f"{name:<18} {acc:>9.2f}%")

names = list(model_registry.keys())
accs  = [test_accs.get(n, 0) * 100 for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, accs, color='steelblue')
ax.set_ylim(0, 105)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Model Comparison — Test Accuracy')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

---
## Save Best Model Weights

In [ ]:
os.makedirs(WEIGHTS_DIR, exist_ok=True)

best_model_name = max(test_accs, key=test_accs.get)
best_model_acc  = test_accs[best_model_name]

print(f'Best model : {best_model_name}')
print(f'Test acc   : {best_model_acc * 100:.2f}%')

save_path = os.path.join(WEIGHTS_DIR, f'{best_model_name}_best.pt')
torch.save({
    'model_name': best_model_name,
    'test_acc':   best_model_acc,
    'dropout':    DROPOUT,
    'state_dict': best_states[best_model_name],
}, save_path)

print(f'Saved → {save_path}')

---
## Retrain Best Model on Full Dataset

Loads the best checkpoint and retrains on **all** available data (no held-out test set).
Use this to produce final deployment weights after evaluation is complete.

Early stopping monitors training loss (no validation set available at this stage).

In [9]:
RETRAIN_MAX_EPOCHS = 15
RETRAIN_PATIENCE   = 3
RETRAIN_MIN_DELTA  = 0.002
RETRAIN_LR         = 1e-3

# Change this if you want to retrain a specific model rather than the best from above
# CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, f'{best_model_name}_best.pt')
CHECKPOINT_PATH = "/Users/chrisdollo/Documents/Research/putEMG prime/formats/deep_learning_approach/model/weights/EMG_TCN_best.pt"

In [11]:
model_registry_build = {
    'EEGNet':         lambda d: models.EEGNet(dropout_rate=d),
    'ShallowConvNet': lambda d: models.ShallowConvNet(dropout_rate=d),
    'DeepConvNet':    lambda d: models.DeepConvNet(dropout_rate=d),
    'CNN_LSTM':       lambda d: models.CNN_LSTM(dropout_rate=d),
    'EMG_TCN':        lambda d: models.EMG_TCN(dropout_rate=d),
}

checkpoint  = torch.load(CHECKPOINT_PATH, map_location=device)
model_name  = checkpoint['model_name']
dropout     = checkpoint['dropout']
eval_acc    = checkpoint['test_acc']

print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Model      : {model_name}')
print(f'Eval acc   : {eval_acc * 100:.2f}%')

retrain_model = model_registry_build[model_name](dropout).to(device)
retrain_model.load_state_dict(checkpoint['state_dict'])

optimizer = optim.Adam(retrain_model.parameters(), lr=RETRAIN_LR)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
criterion = nn.CrossEntropyLoss()

best_loss  = float('inf')
best_state = None
bad_epochs = 0

for epoch in range(RETRAIN_MAX_EPOCHS):
    print("Here")
    tr_loss = train(retrain_model, full_loader, criterion, optimizer, device)
    tr_acc  = evaluate(retrain_model, full_loader, device)
    curr_lr = optimizer.param_groups[0]['lr']

    if tr_loss < best_loss:
        best_loss  = tr_loss
        best_state = {k: v.detach().cpu().clone() for k, v in retrain_model.state_dict().items()}

    scheduler.step(tr_loss)

    if tr_loss <= (best_loss + RETRAIN_MIN_DELTA):
        bad_epochs = 0
    else:
        bad_epochs += 1

    print(f'  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  acc={tr_acc*100:.2f}%  '
          f'best_loss={best_loss:.4f}  lr={curr_lr:.2e}')

    if bad_epochs >= RETRAIN_PATIENCE:
        print(f'  Early stopping at epoch {epoch+1}.')
        break

retrain_model.load_state_dict(best_state)
print(f'\nRetrain complete. Best training loss: {best_loss:.4f}')

Checkpoint : /Users/chrisdollo/Documents/Research/putEMG prime/formats/deep_learning_approach/model/weights/EMG_TCN_best.pt
Model      : EMG_TCN
Eval acc   : 93.57%
Here


KeyboardInterrupt: 

In [ ]:
final_save_path = os.path.join(WEIGHTS_DIR, f'{model_name}_final.pt')
torch.save({
    'model_name': model_name,
    'dropout':    dropout,
    'eval_acc':   eval_acc,
    'state_dict': best_state,
}, final_save_path)

print(f'Final weights saved → {final_save_path}')